In [ ]:
# RSNA Knee 2-ENCODER ENSEMBLE: soup (full-frame) + crop-FT (150mm crop). K=18 each, blend 0.4 soup / 0.6 crop.
# Oracle gpt56: soup 0.8720, crop-FT 0.8795, ENSEMBLE(0.4/0.6) 0.8868. crop-FT-solo K40 = LB 0.924.
import os, glob, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torchvision.models import resnet50
import pydicom, cv2
torch.set_num_threads(os.cpu_count() or 4)
K=40; IMG=384; RES=224; EMB_BS=128; CROP_MM=150.0; W_SOUP=0.4; W_CROP=0.6
LABELS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA',
        'Effusion','Synovitis',"Baker's",'Contusion','Fracture']
dev='cuda' if torch.cuda.is_available() else 'cpu'
print('torch',torch.__version__,'| device',dev,'| K',K,'| blend',W_SOUP,W_CROP)
base=None
for r in sorted(glob.glob('/kaggle/input/*')):
    h=glob.glob(r+'/**/test_series.csv', recursive=True)
    if h: base=os.path.dirname(h[0]); break
print('base',base)
test_series=pd.read_csv(os.path.join(base,'test_series.csv'))
sub=pd.read_csv(os.path.join(base,'sample_submission.csv'))
print('test studies', sub.shape[0])

In [ ]:
class Enc(nn.Module):
    def __init__(s):
        super().__init__(); s.backbone=nn.Sequential(*list(resnet50(weights=None).children())[:-2])
    def forward(s,x): return s.backbone(x).mean(dim=(2,3))
def load_enc(fn):
    p=glob.glob('/kaggle/input/**/'+fn, recursive=True)[0]
    e=Enc(); e.load_state_dict(torch.load(p, map_location='cpu'), strict=True); return e.eval().to(dev), p
soup_enc, sp = load_enc('soup_enc.pt'); crop_enc, cp = load_enc('cropft_enc.pt')
print('encoders:', sp, '|', cp)
class MIL(nn.Module):
    def __init__(s,di=2048,d=256,n=12):
        super().__init__(); s.proj=nn.Sequential(nn.Linear(di,d),nn.LayerNorm(d),nn.GELU())
        s.plane=nn.Embedding(3,d); s.fluid=nn.Embedding(2,d)
        s.V=nn.Linear(d,128); s.U=nn.Linear(d,128); s.w=nn.Linear(128,1)
        s.head=nn.Sequential(nn.Dropout(0.3), nn.Linear(d,n))
    def forward(s,x,pid,fb):
        h=s.proj(x)+s.plane(pid)+s.fluid(fb)
        a=torch.softmax(s.w(torch.tanh(s.V(h))*torch.sigmoid(s.U(h))),0)
        return s.head((a*h).sum(0))
def load_heads(fn):
    p=glob.glob('/kaggle/input/**/'+fn, recursive=True)[0]; ck=torch.load(p, map_location='cpu'); DI=int(ck.get('di',2048))
    hs=[]
    for sd in ck['heads']:
        m=MIL(di=DI).to(dev).eval(); m.load_state_dict(sd); hs.append(m)
    return hs
soup_heads=load_heads('head_weights_soup.pt'); crop_heads=load_heads('head_weights_cropft.pt')
print('heads: soup',len(soup_heads),'crop',len(crop_heads))
PL={'Sagittal':0,'Coronal':1,'Axial':2}
print('ready')

In [ ]:
def sampled(sd,k):
    fs=sorted(glob.glob(sd+'/*.dcm')); n=len(fs)
    if n==0: return []
    if n<=k: return fs
    lo,hi=int(n*.12),int(n*.88)
    if hi<=lo: lo,hi=0,n
    return [fs[int(round(x))] for x in np.linspace(lo,hi-1,k)]
def slc2(p):
    try:
        d=pydicom.dcmread(p); a=d.pixel_array.astype(np.float32)
        a=a*float(getattr(d,'RescaleSlope',1) or 1)+float(getattr(d,'RescaleIntercept',0) or 0)
        # full-frame
        lo,hi=np.percentile(a,0.5),np.percentile(a,99.5); full=np.clip((a-lo)/max(hi-lo,1e-6),0,1)
        full=cv2.resize(full,(IMG,IMG),interpolation=cv2.INTER_AREA).astype(np.float32)
        # 150mm center crop (match crop-FT training)
        ac=a; ps=getattr(d,'PixelSpacing',None)
        if ps is not None:
            try:
                psr=float(ps[0]); psc=float(ps[1]); H,W=a.shape
                cr=min(int(round(CROP_MM/psr)),H); cc=min(int(round(CROP_MM/psc)),W)
                r0=max(0,(H-cr)//2); c0=max(0,(W-cc)//2); ac=a[r0:r0+cr, c0:c0+cc]
            except Exception: pass
        lo,hi=np.percentile(ac,0.5),np.percentile(ac,99.5); crop=np.clip((ac-lo)/max(hi-lo,1e-6),0,1)
        crop=cv2.resize(crop,(IMG,IMG),interpolation=cv2.INTER_AREA).astype(np.float32)
        return full, crop
    except Exception: return None, None
@torch.no_grad()
def embed(imgs, enc):
    x=torch.from_numpy(np.stack(imgs))[:,None]; x=x*2.0-1.0; x=x.expand(-1,3,-1,-1).contiguous()
    if x.shape[-1]!=RES: x=F.interpolate(x,size=(RES,RES),mode='bilinear',align_corners=False)
    return enc(x.to(dev)).float()
@torch.no_grad()
def predict_study(study):
    fulls=[];crops=[];pids=[];fbs=[]
    for _,sr in test_series[test_series['StudyInstanceUID']==study].iterrows():
        sd=os.path.join(base,'test_series',study,sr['SeriesInstanceUID'])
        if not os.path.isdir(sd):
            alt=glob.glob(os.path.join(base,'**',sr['SeriesInstanceUID']),recursive=True); sd=alt[0] if alt else None
        if not sd: continue
        pid=PL.get(sr['Anatomical_Plane'],0); fb=int(sr['Fluid_Sensitive'])
        for p in sampled(sd,K):
            f,c=slc2(p)
            if f is not None: fulls.append(f);crops.append(c);pids.append(pid);fbs.append(fb)
    if not fulls: return np.full(12,0.5)
    pid=torch.tensor(pids,device=dev); fb=torch.tensor(fbs,device=dev)
    Ef=[];Ec=[]
    for i in range(0,len(fulls),EMB_BS): Ef.append(embed(fulls[i:i+EMB_BS],soup_enc)); Ec.append(embed(crops[i:i+EMB_BS],crop_enc))
    Xf=torch.cat(Ef,0); Xc=torch.cat(Ec,0)
    ps=np.mean([torch.sigmoid(h(Xf,pid,fb)).cpu().numpy() for h in soup_heads],0)
    pc=np.mean([torch.sigmoid(h(Xc,pid,fb)).cpu().numpy() for h in crop_heads],0)
    return W_SOUP*ps + W_CROP*pc
print('fns ready')

In [ ]:
ids=list(sub['StudyInstanceUID']); t0=time.time(); out={}
for i,s in enumerate(ids):
    out[s]=predict_study(s)
    if i%100==0: print(f'{i}/{len(ids)} {time.time()-t0:.0f}s',flush=True)
res=pd.DataFrame({'StudyInstanceUID':ids})
for j,l in enumerate(LABELS): res[l]=[out[s][j] for s in ids]
res=res[['StudyInstanceUID']+LABELS]; res.to_csv('submission.csv',index=False)
print('WROTE submission.csv',res.shape,f'{time.time()-t0:.0f}s'); print(res.head())